<h1 style = "color : #0EE071; text-align : center;"><em>Where should I live?</em> - Data Science in Action Notebook</h1>
<p style = "font-size : 16px; text-align: center;">
In this final phase, you will bring everything together by integrating your preprocessed dataset and transforming it into <code>meaningful insights or tools</code>. This is
your opportunity to be creative: choose techniques you find most appropriate and
develop something that would <code>genuinely help people compare European cities
and decide where to live</code>.</p>
<br>
<p style = "font-size : 12px; text-align: center;"><b>NOVA IMS</b></p>
<p style = "font-size : 10px; text-align: center;">Programming for Data Science</p>
<p style = "font-size : 10px; text-align: center;">Diogo Gonçalves, João Marques, Juan Mendes & Gustavo Franco</p>
<br>

<h2  style = "color : #0EE071;"> Imports</h2>

In [2]:
#!pip install panel

In [38]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
from bs4 import BeautifulSoup
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys
import requests
import re
import pandas as pd
import panel as pn
import time
import plotly.express as px
import numpy as np
import panel as pn
import pandas as pd
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

<hr style = "border: 3px solid #0EE071;">
<h2 style = "color : #0EE071;">Dataset Importing </h2>
<p style = "font-size : 15px;">Reading of dataset from <code>city_data_clean.csv</code> file</p>
<p>The dataset is in a <code>.csv</code> file, and uses <code>,</code> as a separator.</p>

In [39]:
city_data = pd.read_csv("city_data_clean.csv", sep = ",")

<hr style = "border: 3px solid #0EE071;">
<h2  style = "color : #0EE071;">Web Scraping</h2>
<p style="font-size: 15px;">
  <span style="font-size: 20px;">Short summary:</span>
  <br><br>
  - <code>requests</code> : download HTML only<br>
  - <code>BeautifulSoup</code> : extract info from that HTML<br>
  - <code>Selenium</code> : control a browser and interact with the page
</p>

In [40]:
url_dictionary = {"Food Prices": "https://www.numbeo.com/food-prices/in/city",
"Gas Prices Calculator" : "https://www.numbeo.com/gas-prices/in/city",
"Salary Calculator" : "https://www.numbeo.com/cost-of-living/prices_by_country.jsp?itemId=105&displayCurrency=EUR",
"Quality of Life": "https://www.numbeo.com/quality-of-life/in/city",
"Crime": "https://www.numbeo.com/crime/in/city",
"Pollution": "https://www.numbeo.com/pollution/in/city"}

for element in url_dictionary:
    city_data.loc[:, element] = None

In [52]:
def food_prices(readable_html):
    rows = readable_html.find_all("tr", class_=["tr_standard", "tr_highlighted"])
    data = []
    for row in rows:
        tds = row.find_all("td")
        item = tds[0].get_text(strip=True).replace("(", "").replace(")", "")
        price = tds[1].get_text(strip=True).replace("\xa0", "").replace("(", "").replace(")", "")
        data.append([item, price])
    return data

def gas_prices(readable_html):
    return readable_html.find("span", class_="first_currency").text

def quality_of_life(readable_html):
    return [element.text.replace("\n", "") for element in readable_html.find_all("td", style ="text-align: right")], [element.text.replace("&nbsp;", "").replace("\xa0\n", "") for element in readable_html.find_all("td", style ="text-align: center; font-weight: 600")]

def crime(readable_html):
    return [''.join(digit for digit in element.get_text(strip=True) if digit.isdigit() or digit == '.') for element in readable_html.find_all("td", style="text-align: right", class_="indexValueTd")], [element.text.replace("\n", "") for element in readable_html.find_all("span", class_ ="green_light_standard")]

def pollution(readable_html):
        return [''.join(digit for digit in element.get_text(strip=True) if digit.isdigit() or digit == '.') for element in readable_html.find_all("td", style="text-align: right", class_="indexValueTd")], [element.text.replace("\n", "") for element in readable_html.find_all("span", class_ ="green_light_standard")]

def salary(readable_html):
    first_list = [x.strip() for x in readable_html.body.text.split("(After Tax)")[4].split("Last Update")[0].split("\n") if x.strip() != ""]
    result = []
    for item in first_list:
        country = re.match(r"^[^\d]+", item).group().strip()
        number = float(re.search(r"\d+\.\d+", item).group())
        result.append([country, number])
    return result
    
def get_info(data):
    for index in data.index:
        city_name = data.loc[index, "City"]
        for char in url_dictionary:
            url = url_dictionary[char].replace("city", city_name)
            html = requests.get(url)
            readable_html = BeautifulSoup(html.text, "html.parser")

            if char == "Food Prices":
                try:
                    data.loc[index, char] = str(food_prices(readable_html))

                except:
                    data.loc[index, char] = None

            elif char == "Gas Prices Calculator":
                try:
                    data.loc[index, char] = gas_prices(readable_html)

                except:
                    data.loc[index, char] = None

            elif char == "Salary Calculator":
                try:
                    data.loc[index, char] = str(salary(readable_html))
                except:
                    data.loc[index, char] = None
            elif char == "Quality of Life":
                try:
                    data.loc[index, char] = str(quality_of_life(readable_html))
                except:
                    data.loc[index, char] = None

            elif char == "Crime":
                try:
                    data.loc[index, char] = str(crime(readable_html))

                except:
                    data.loc[index, char] = None
            elif char == "Pollution":
                try:
                    data.loc[index, char] = str(pollution(readable_html))
                except:
                    data.loc[index, char] = None

In [53]:
get_info(city_data)

In [54]:
city_data

,Population Density,Population,Working Age Population,Youth Dependency Ratio,Unemployment Rate,GDP per Capita,Days of very strong heat stress,Average Monthly Salary,Average Rent Price,Average Cost of Living,...,Turkish,Unknown,Urdu,Valencian,Food Prices,Gas Prices Calculator,Salary Calculator,Quality of Life,Crime,Pollution
0,310.0,2983513,2018818,20.1,10.2,55770.0,3,2500,1050,2061,...,1,0,0,0,"[['Milk Regular, 0.25 Liter', '0.37€'], ['Fres...",1.55 €,"[['Kazakhstan', 503.16], ['Portugal', 1117.69]...","(['209.00', '134.70', '71.77', '81.85', '81.77...","(['27.19', '56.82', '25.03', '24.02', '15.83',...","(['15.50', '4.15', '11.33', '14.12', '30.14', ..."
1,243.0,375489,250472,20.3,3.0,66689.0,0,3200,1100,2186,...,0,0,0,0,"[['Milk Regular, 0.25 Liter', '0.38€'], ['Fres...",1.50 €,"[['Kazakhstan', 503.16], ['Portugal', 1117.69]...","(['190.37', '117.65', '79.50', '73.25', '76.51...","(['11.70', '57.33', '16.59', '18.14', '7.67', ...","(['18.06', '14.29', '19.64', '25.00', '41.07',..."
2,681.0,3284548,2137425,27.5,10.7,62500.0,3,3350,1200,1900,...,0,0,0,0,"[['Milk Regular, 0.25 Liter', '0.33€'], ['Fres...",1.64 €,"[['Kazakhstan', 503.16], ['Portugal', 1117.69]...","(['155.57', '124.52', '43.92', '73.57', '83.85...","(['62.04', '71.04', '50.67', '57.56', '44.74',...","(['63.00', '37.09', '57.11', '64.84', '55.83',..."
3,928.0,1139663,723396,27.7,6.2,57595.0,3,2609,900,1953,...,0,0,0,0,"[['Milk Regular, 0.25 Liter', '0.28€'], ['Fres...",1.63 €,"[['Kazakhstan', 503.16], ['Portugal', 1117.69]...","(['165.61', '116.92', '60.06', '79.77', '85.42...","(['44.05', '61.65', '36.48', '35.91', '25.64',...","(['63.16', '20.28', '39.62', '49.52', '60.38',..."
4,552.0,645813,417832,24.8,5.3,53311.0,2,2400,827,1200,...,0,0,0,0,"[['Milk Regular, 0.25 Liter', '0.26€'], ['Fres...",1.61 €,"[['Kazakhstan', 503.16], ['Portugal', 1117.69]...","(['205.47', '122.06', '75.81', '82.64', '88.72...","(['25.74', '52.30', '23.92', '20.47', '13.14',...","(['25.47', '10.00', '22.00', '24.00', '29.00',..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,334.0,2344124,1534225,28.5,6.2,70950.0,0,2700,1400,2300,...,0,0,0,0,"[['Milk Regular, 0.25 Liter', '4.20kr'], ['Fre...",17.16 kr,"[['Kazakhstan', 503.16], ['Portugal', 1117.69]...","(['174.17', '119.11', '53.47', '65.71', '69.67...","(['52.84', '71.72', '39.23', '44.77', '34.83',...","(['15.10', '7.82', '20.25', '25.17', '27.64', ..."
80,245.0,1037675,672152,28.2,6.3,49588.0,0,2500,1200,2100,...,0,0,0,0,"[['Milk Regular, 0.25 Liter', '4.04kr'], ['Fre...",17.59 kr,"[['Kazakhstan', 503.16], ['Portugal', 1117.69]...","(['204.75', '155.83', '54.17', '69.93', '77.49...","(['53.61', '72.22', '35.36', '43.50', '32.73',...","(['16.67', '6.62', '13.81', '27.65', '30.30', ..."
81,368.0,680335,436271,29.4,9.2,44387.0,0,2400,1100,2000,...,0,0,0,0,"[['Milk Regular, 0.25 Liter', '4.12kr'], ['Fre...",18.08 kr,"[['Kazakhstan', 503.16], ['Portugal', 1117.69]...","(['191.01', '128.99', '44.97', '68.05', '82.30...","(['62.09', '68.65', '46.75', '54.98', '44.78',...","(['12.50', '5.88', '16.91', '27.21', '25.00', ..."
82,1922.0,4843511,3417691,30.0,14.4,38916.0,3,900,450,900,...,1,0,0,0,"[['Milk Regular, 0.25 Liter', '10.26TL'], ['Fr...",51.16 TL,"[['Kazakhstan', 503.16], ['Portugal', 1117.69]...","(['140.79', '62.79', '60.49', '70.03', '91.49'...","(['39.05', '68.43', '35.57', '34.36', '26.94',...","(['59.80', '53.63', '40.77', '50.73', '50.44',..."


In [55]:
def get_coordenates_insert(data):
    # Open Google Chrome and search Wikipedia
    browser = webdriver.Chrome()
    browser.get('https://en.wikipedia.org/wiki/Main_Page')
    time.sleep(1)

    # Loop to insert the coordenates in the dataset
    for idx in data.index:
        city = data.loc[idx, "City"]
        country = data.loc[idx, "Country"]

        result = get_coordinates(browser ,city, country)
        if result:
            latitude, longitude = result
        else:
            latitude, longitude = None, None

        data.loc[idx, "Latitude"] = latitude
        data.loc[idx, "Longitude"] = longitude

    browser.quit()

In [56]:
get_coordenates_insert(city_data)

NameError: name 'get_coordinates' is not defined

<hr style = "border: 3px solid #0EE071;">
<h2 style = "color : #0EE071;">Dashboard</h2>
<p style = "font-size : 16px;">It displays our data interactively through a dashboard</p>
<br>

Import of Images

In [16]:
country_flag_urls = {
    "Austria": "https://flagcdn.com/w160/at.png",
    "Belgium": "https://flagcdn.com/w160/be.png",
    "Bulgaria": "https://flagcdn.com/w160/bg.png",
    "Switzerland": "https://flagcdn.com/w160/ch.png",
    "Cyprus": "https://flagcdn.com/w160/cy.png",
    "Czechia": "https://flagcdn.com/w160/cz.png",
    "Germany": "https://flagcdn.com/w160/de.png",
    "Denmark": "https://flagcdn.com/w160/dk.png",
    "Spain": "https://flagcdn.com/w160/es.png",
    "Estonia": "https://flagcdn.com/w160/ee.png",
    "Finland": "https://flagcdn.com/w160/fi.png",
    "France": "https://flagcdn.com/w160/fr.png",
    "United Kingdom": "https://flagcdn.com/w160/gb.png",
    "Greece": "https://flagcdn.com/w160/gr.png",
    "Croatia": "https://flagcdn.com/w160/hr.png",
    "Hungary": "https://flagcdn.com/w160/hu.png",
    "Ireland": "https://flagcdn.com/w160/ie.png",
    "Italy": "https://flagcdn.com/w160/it.png",
    "Luxembourg": "https://flagcdn.com/w160/lu.png",
    "Latvia": "https://flagcdn.com/w160/lv.png",
    "Malta": "https://flagcdn.com/w160/mt.png",
    "Netherlands": "https://flagcdn.com/w160/nl.png",
    "Norway": "https://flagcdn.com/w160/no.png",
    "Poland": "https://flagcdn.com/w160/pl.png",
    "Portugal": "https://flagcdn.com/w160/pt.png",
    "Romania": "https://flagcdn.com/w160/ro.png",
    "Slovak Republic": "https://flagcdn.com/w160/sk.png",
    "Slovenia": "https://flagcdn.com/w160/si.png",
    "Sweden": "https://flagcdn.com/w160/se.png",
    "Turkiye": "https://flagcdn.com/w160/tr.png"
}

In [22]:
city_data["Image Url"] = None
for index in city_data.index:
    for country, url in country_flag_urls.items():
        if country in city_data.loc[index, "Country"]:
            city_data.loc[index, "Image Url"] = url
            break

In [23]:
city_data

,Population Density,Population,Working Age Population,Youth Dependency Ratio,Unemployment Rate,GDP per Capita,Days of very strong heat stress,Average Monthly Salary,Average Rent Price,Average Cost of Living,...,Unknown,Urdu,Valencian,Food Prices,Gas Prices Calculator,Salary Calculator,Quality of Life,Crime,Pollution,Image Url
0,310.0,2983513,2018818,20.1,10.2,55770.0,3,2500,1050,2061,...,0,0,0,[],None,None,"([], [])","([], [])","([], [])",https://flagcdn.com/w160/at.png
1,243.0,375489,250472,20.3,3.0,66689.0,0,3200,1100,2186,...,0,0,0,[],None,None,"([], [])","([], [])","([], [])",https://flagcdn.com/w160/at.png
2,681.0,3284548,2137425,27.5,10.7,62500.0,3,3350,1200,1900,...,0,0,0,[],None,None,"([], [])","([], [])","([], [])",https://flagcdn.com/w160/be.png
3,928.0,1139663,723396,27.7,6.2,57595.0,3,2609,900,1953,...,0,0,0,[],None,None,"([], [])","([], [])","([], [])",https://flagcdn.com/w160/be.png
4,552.0,645813,417832,24.8,5.3,53311.0,2,2400,827,1200,...,0,0,0,[],None,None,"([], [])","([], [])","([], [])",https://flagcdn.com/w160/be.png
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,334.0,2344124,1534225,28.5,6.2,70950.0,0,2700,1400,2300,...,0,0,0,[],None,None,"([], [])","([], [])","([], [])",https://flagcdn.com/w160/se.png
80,245.0,1037675,672152,28.2,6.3,49588.0,0,2500,1200,2100,...,0,0,0,[],None,None,"([], [])","([], [])","([], [])",https://flagcdn.com/w160/se.png
81,368.0,680335,436271,29.4,9.2,44387.0,0,2400,1100,2000,...,0,0,0,[],None,None,"([], [])","([], [])","([], [])",https://flagcdn.com/w160/se.png
82,1922.0,4843511,3417691,30.0,14.4,38916.0,3,900,450,900,...,0,0,0,[],None,None,"([], [])","([], [])","([], [])",https://flagcdn.com/w160/tr.png


In [37]:
city_data.loc[36, "Country"]

KeyError: 36

Dashboard

In [36]:
# Initialize Panel extension
pn.extension('plotly')

# ======================================
# DATA PREPARATION
# ======================================
# Clean column names and fill NA values
city_data.columns = [col.strip() for col in city_data.columns]
city_data = city_data.fillna(0)

# ======================================
# STYLES
# ======================================
dashboard_style = {'background': '#F7F9FC', 'padding': '25px', 'border-radius': '15px'}
header_style = {'background': '#2A3B4D', 'padding': '5px 25px', 'border-radius': '10px', 'margin': '0 0 20px 0'}
card_style = {'background': '#FFFFFF', 'border': '1px solid #E0E0E0', 'border-radius': '10px', 'padding': '25px', 'margin': '10px 0'}
slider_style = {'background': '#FFFFFF', 'border-radius': '8px', 'padding': '15px', 'margin': '8px 0', 'box-shadow': '0 2px 12px rgba(0, 0, 0, 0.05)', 'border-left': '4px solid #0D6EFD',  'border-top': '1px solid #E0E0E0', 'border-right': '1px solid #E0E0E0', 'border-bottom': '1px solid #E0E0E0'}
infobox_style = {'background': '#FFFFFF', 'border-left': '5px solid #0D6EFD', 'border-radius': '8px', 'padding': '10px', 'margin': '8px', 'box-shadow': '0 2px 6px 0 rgba(0,0,0,0.07)', 'width': '260px', 'height': '80px', 'box-sizing': 'border-box', 'text-align': 'center', 'display': 'flex', 'flex-direction': 'column', 'justify-content': 'center'}

# ======================================
# WIDGETS
# ======================================
# City selectors
country_1 = pn.widgets.Select(name="City", options=sorted(list(city_data["City"])), min_width=200, width_policy='max', sizing_mode='stretch_width')
country_2 = pn.widgets.Select(name="City", options=list(city_data["City"]), min_width=200, width_policy='max', sizing_mode='stretch_width')

# Characteristic selector
characteristic_selector = pn.widgets.MultiChoice(name="Select Characteristics", options=['Population Density', 'Population', 'Working Age Population',
       'Youth Dependency Ratio', 'Unemployment Rate', 'GDP per Capita',
       'Days of very strong heat stress', 'Average Monthly Salary',
       'Average Rent Price', 'Average Cost of Living',
       'Average Price Groceries','Food Prices', 'Gas Prices Calculator', 'Salary Calculator',
       'Quality of Life', 'Crime', 'Pollution'], value=[], max_items=23, min_width=300, placeholder="Click to select characteristics")

# View selector
select = pn.widgets.RadioButtonGroup(name="Select", options=["Characteristics", "Compare Cities"],  button_type='primary', button_style='outline', min_width=200, sizing_mode='stretch_width')

# Navigation buttons
big_button = pn.widgets.Button(name="Your Country Is...", button_type='primary', height=40, styles={ 'font-size': '16px', 'font-weight': 'bold', 'padding': '5px'}, sizing_mode='stretch_width', width_policy='max')
go_back_button = pn.widgets.Button(name='Go Back', button_type='default', sizing_mode='stretch_width')

# ======================================
# DATA PROCESSING FUNCTIONS
# ======================================
def scale(data):
    """Scale numeric columns for comparison"""
    for col in data.select_dtypes("number").columns:
        if "Scaled" not in col:
            data[f"{col} Scaled"] = data[col] / data[col].sum()

# Apply scaling
scale(city_data)

def best_country(sliders, data):
    """Calculate best matching country based on slider weights"""
    final = pd.Series(0, index=data["City"], name="Scores")
    for name, slider in sliders.items():
        col = f"{name} Scaled"
        final += slider.value * data[col]
    final = final.sort_values(ascending=False)
    return final.idxmax(), final

def update_display(*_):
    """Update display when sliders change"""
    best, final = best_country(sliders, city_data)
    return best, final

# ======================================
# UI COMPONENTS
# ======================================
def binary(option):
    """Render either single city or comparison view based on selection"""
    def get_image(city):
        row = city_data[city_data["City"] == city]
        if row.empty:
            return None
        return pn.pane.PNG(row.iloc[0]["Image Url"], width=240, height=160)

    interative_image = pn.bind(get_image, country_1)  
    
    if option == "Characteristics":
        return pn.Column(pn.pane.Markdown("### Your most wanted country"), interative_image, styles=card_style)
    else:
        return pn.Column(pn.pane.Markdown("### Choose your country"), pn.Column(country_1, country_2), styles=card_style)

def chars_left(selected_chars, country):
    """Display selected characteristics for a country"""
    global boards
    boards = []
    
    emoji_map = {
        "Population Density": "👨‍👩‍👧", 
        "Population": "🌎", 
        "Working Age Population": "💼🔞", 
        "Youth Dependency Ratio": "👶🏻", 
        "Unemployment Rate": "💼📉", 
        "GDP per Capita": "💰", 
        "Days of very strong heat stress": "🌡️🔥", 
        "Main Spoken Languages": "💬🌐", 
        "Average Monthly Salary": "💼💰", 
        "Avgerage Rent Price": "🏠💲", 
        "Average Cost of Living": "💸👨‍👨‍👧‍👦", 
        "Average Price Groceries": "💲🍽️", 
        "Last Data Update": "🔄"
    }
    
    for char in selected_chars:
        if char in emoji_map:
            emoji = emoji_map[char]
            board = pn.pane.Markdown(f"<div style='font-size:8pt'><b>{char}</b> {emoji}</div><br><div style='font-size:7pt'>Current value: <b>{data_copy.loc[country, char]}</b></div>", styles=infobox_style)
            boards.append(board)

    # Create rows of 3 columns
    rows = []
    for i in range(0, len(boards), 3):
        rows.append(pn.Row(*boards[i:i+3]))
    
    return pn.Column(*rows)

def all_chars(chars, country_1, country_2):
    """Display comparison of all characteristics between two countries"""
    global boards
    boards = []
    
    emoji_map = {
        "Population Density": "👨‍👩‍👧", 
        "Population": "🌎", 
        "Working Age Population": "💼🔞", 
        "Youth Dependency Ratio": "👶🏻", 
        "Unemployment Rate": "💼📉", 
        "GDP per Capita": "💰", 
        "Days of very strong heat stress": "🌡️🔥", 
        "Main Spoken Languages": "💬🌐", 
        "Average Monthly Salary": "💼💰", 
        "Avgerage Rent Price": "🏠💲", 
        "Average Cost of Living": "💸👨‍👨‍👧‍👦", 
        "Average Price Groceries": "💲🍽️", 
        "Last Data Update": "🔄"
    }
    
    # Get image URLs
    row1 = city_data[city_data["City"] == country_1]
    row2 = city_data[city_data["City"] == country_2]

    img1_url = row1.iloc[0]["Image Url"]
    img2_url = row2.iloc[0]["Image Url"]
    
    for char in chars:
        if char in emoji_map:
            # Create image panes with proper HTML
            image1 = pn.pane.HTML(f'<img src="{img1_url}" width="30" height="20" style="vertical-align:middle">')
            image2 = pn.pane.HTML(f'<img src="{img2_url}" width="30" height="20" style="vertical-align:middle">')
            emoji = emoji_map[char]
            
            comparison_row = pn.Row(
                image1,
                pn.pane.HTML(f"<div style='text-align:center; font-size:9px'><b>{city_data.loc[country_1, char]}</b></div>"),
                pn.pane.HTML("<div style='text-align:center; font-size:10px'><b>vs</b></div>"),
                pn.pane.HTML(f"<div style='text-align:center; font-size:9px'><b>{city_data.loc[country_2, char]}</b></div>"),
                image2, align='center')
        
            board = pn.Column(pn.Column(pn.pane.Markdown(f"<div style='text-align:center; font-size:12px'><b>{char} {emoji}</b></div>"), align='center'), comparison_row, styles=infobox_style, align='center')
            
            boards.append(board)

    # Create rows of 3 columns
    rows = []
    for i in range(0, len(boards), 3):
        rows.append(pn.Row(*boards[i:i+3]))
    
    return pn.Column(*rows)

def create_pie_chart(*values):
    """Create a pie chart showing weight distribution"""
    all_labels = list(sliders.keys())
    values_list = []
    labels_list = []
    
    for i in range(len(values)):
        if values[i] > 0:
            values_list.append(values[i])
            labels_list.append(all_labels[i])
    
    if not values_list:
        return pn.pane.Markdown("### Adjust the sliders to see the weight distribution")

    n = len(values_list)
    blue_colors = [f'rgb({int(30 + i * (225/n))}, {int(100 + i * (155/n))}, {int(180 + i * (75/n))})' for i in range(n)]
    
    pie = go.Pie(labels=labels_list, values=values_list, textinfo='label+percent', hole=0.3, textfont_size=12, textposition='inside', marker_colors=blue_colors)
    
    fig = go.Figure(data=[pie])
    
    fig.update_layout(margin=dict(l=10, r=10, t=30, b=10), showlegend=False, height=250, width=400, xaxis=dict(showgrid=False, zeroline=False), yaxis=dict(showgrid=False, zeroline=False))
    
    return pn.pane.Plotly(fig, config={'displayModeBar': False}, width=400, height=250, sizing_mode='fixed')

# ======================================
# SLIDERS CREATION
# ======================================
sliders = {}
for col in city_data.columns:
    if col in ['Population Density', 'Population', 'Working Age Population',
       'Youth Dependency Ratio', 'Unemployment Rate', 'GDP per Capita',
       'Days of very strong heat stress', 'Average Monthly Salary',
       'Average Rent Price', 'Average Cost of Living',
       'Average Price Groceries','Food Prices', 'Gas Prices Calculator', 'Salary Calculator',
       'Quality of Life', 'Crime', 'Pollution']:
        
        sliders[col] = pn.widgets.IntSlider(name=col, start=0, end=10, step=1, width=180, styles=slider_style, width_policy='max')

# Create 3 columns for sliders
slider_columns = [[], [], []]
for i, (col, slider) in enumerate(sliders.items()):
    column_index = i % 3
    slider_columns[column_index].append(slider)

# Create Rows for each column of sliders
slider_rows = []
for col_sliders in slider_columns:
    if col_sliders:
        slider_rows.append(pn.Column(*col_sliders, width=190))

# Create sliders panel
sliders_panel = pn.Column(
    pn.pane.Markdown("### On a scale from one to ten, how important are these characteristics?", styles={'color': '#0D6EFD'}), pn.Row(*slider_rows), styles={'background': '#FFFFFF', 'padding': '15px', 'border-radius': '8px'}, width_policy='max')

# Initialize slider values
slider_values = [slider.param.value for slider in sliders.values()]

# Create the pie chart panel
pie_chart = pn.panel(pn.bind(create_pie_chart, *slider_values), config={'displayModeBar': False})

# ======================================
# LAYOUTS
# ======================================
header = pn.Row(pn.pane.Markdown("## Dashboard", styles={'color': '#FFFFFF'}), styles=header_style, sizing_mode='stretch_width')

def one_country():
    """Layout for single country view"""
    header = pn.Row(pn.pane.Markdown("## Dashboard", styles={'color': '#FFFFFF'}), styles=header_style, sizing_mode='stretch_width')
    under_header = pn.Row(select, styles=card_style, width=413)
    left = pn.Column(pn.panel(interactive_binary), styles=card_style, width=373, sizing_mode='fixed')
    right = pn.Column(pn.pane.Markdown("### Select the characteristics that can't really miss"), characteristic_selector, styles=card_style, width=413)
    return pn.Column( header, pn.Row(pn.Column(under_header, pie_chart), pn.Spacer(width = 150) , sliders_panel), pn.Row(big_button, styles={'background': '#2A3B4D', 'padding': '5px 25px', 'border-radius': '10px', 'margin': '0 0 10px 0'}, sizing_mode='stretch_width'))

def two_countries():
    """Layout for comparing two countries"""
    header = pn.Row(pn.pane.Markdown("## Dashboard", styles={'color': '#FFFFFF'}), styles=header_style, sizing_mode='stretch_width')
    under_header = pn.Row(select, styles=card_style)
    return pn.Column(header, pn.Row(pn.Column(under_header, interactive_binary, sizing_mode='fixed'), pn.Spacer(width = 75) ,interactive_boards_all, styles=card_style))

def layout_choice(selection):
    """Choose between single country or comparison layout"""
    if selection == "Characteristics":
        return one_country()
    else:
        return two_countries()

# ======================================
# INTERACTIVE BINDINGS
# ======================================
# Interaction between the select "Characteristics" and "Comparison of the two cities"
interactive_binary = pn.bind(binary, select)

# Comparison of the comparison of the boxes and the things selected "names" with the slider
interactive_boards = pn.bind(chars_left, characteristic_selector.param.value)

# Comparison of the layout choice function and the select box
interactive_layout = pn.bind(layout_choice, select)

# Get all characteristics for comparison
chars = ['Population Density', 'Population', 'Working Age Population',
       'Youth Dependency Ratio', 'Unemployment Rate', 'GDP per Capita',
       'Days of very strong heat stress', 'Average Monthly Salary',
       'Average Rent Price', 'Average Cost of Living',
       'Average Price Groceries','Food Prices', 'Gas Prices Calculator', 'Salary Calculator',
       'Quality of Life', 'Crime', 'Pollution']

# Bind the all_chars function to city selectors
interactive_boards_all = pn.bind(lambda city1, city2: all_chars(chars, city1, city2), country_1.param.value, country_2.param.value)

# ======================================
# PAGES
# ======================================
def page1():
    """Results page showing best matching country"""
    best, scores = best_country(sliders, city_data)
    best_score = scores[best]

    chars = ['Population Density', 'Population', 'Working Age Population',
       'Youth Dependency Ratio', 'Unemployment Rate', 'GDP per Capita',
       'Days of very strong heat stress', 'Average Monthly Salary',
       'Average Rent Price', 'Average Cost of Living',
       'Average Price Groceries','Food Prices', 'Gas Prices Calculator', 'Salary Calculator',
       'Quality of Life', 'Crime', 'Pollution']

    results = chars_left(chars, best)
    
    def get_image(_):
        row = city_data[city_data["City"] == best]
        return pn.pane.PNG(row.iloc[0]["Image Url"], width=240, height=160)



    return pn.Column(header,pn.Row(pn.Column( pn.pane.Markdown("## Your Perfect Match"), pn.bind(get_image, None), pn.pane.Markdown(f"### {best}"), results), pn.Spacer(width = 100) ,pn.Column(pn.pane.Markdown(f"**Match Score:** {best_score:.2f}"), pn.pane.Markdown("### All Scores:"), pn.pane.DataFrame(scores.to_frame("Score").head(18)))), go_back_button, margin=20)

# ======================================
# NAVIGATION
# ======================================
def show_page(page_name, event=None):
    """Handle page navigation"""
    dashboard.clear()
    
    if page_name == 'home':
        # Always check the current select value
        selection = select.value
        if selection == "Characteristics":
            page = one_country()
        else:
            page = two_countries()
        big_button.on_click(lambda e: show_page('page1'))
        dashboard.append(page)
        
    elif page_name == 'page1':
        page = page1()
        go_back_button.on_click(lambda e: show_page('home'))
        dashboard.append(page)

# ======================================
# INITIALIZATION
# ======================================
# Create dashboard
dashboard = pn.Column(styles=dashboard_style, sizing_mode='stretch_width')

# Set up view selector callback
def on_select_change(event):
    show_page('home')

select.param.watch(on_select_change, 'value')

# Bind the best country display
best_country_display = pn.bind(update_display, *[s.param.value for s in sliders.values()])

# Start with home page
show_page('home')

# ======================================
# SERVE THE DASHBOARD
# ======================================
pn.serve(dashboard)

KeyError: 'City'